# Planning and preferences

In the previous notebook, we have shown that the best fitting model consists in a combination of planning and preferences. We have furher observed an interaction between the entropy of the preferences (i.e. how strong or certain the preference is in a particular state) and the weight of the planning. It would indeed seem that there is a trade off between the two, such that when participants have a strong preference towards a specific action, they tend to rely less on planning and vice versa. 

In this notebook, we will investigate the fit of the model in greater details and conduct additional exploratory analyses to support the idea that the task specific regressors capture the structure of the task leverage by the participants to facilitate their decision. 

# Preparing the data

In [1]:
import pandas as pd
from stabst.MarkovDecisionProcess import MDP
from stabst.TaskConfig import LimitedEnergyTask
import arviz as az
from arviz_stats import bayesian_r2
import bambi as bmb
from stabst.utils import plt_default
import pymc as pm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
plt_default()

# ===================================================================
# Loading the raw data:
data = pd.read_csv("../../data/raw_data/all_participants_data.csv", on_bad_lines='skip')

# ===================================================================
# Data preprocessing:
# Remove nans:
data = data.dropna()
# Remove timeout:
data = data[data["timeout"] == 0]
# Flip responses: 1 = accept:
data["response"] = (data["response"] == 0).astype(int)
# Make trial 1 based
data["trial"] = data["trial"] + 1
# Generate future cost based on the transitions:
transitions_costs = {
    0: [1, 1],
    1: [2, 1],
    2: [1, 2],
    3: [2, 2]
}
data["fc"] = [transitions_costs[row["transition"]][1] for _, row in data.iterrows()]

# ===================================================================
# Task MDP:
# Create the task and its parameters (transition probability, reward...):
task = LimitedEnergyTask(O=[1, 2, 3, 4], p_offer=[1/4] * 4)
task.build()

# Create full MDP and compute solution for later reference:
gamma = 1
task_mdp = MDP(task.states, task.tp, task.r, gamma, s2i=task.s2i)
V_full, Q_full = task_mdp.backward_induction()

# Add decision values to the dataframe:
dv = Q_full[:, 1] - Q_full[:, 0]
# Loop through each trial to set DV:
dv_trials = []
for trial_i, trial in data.iterrows():
    e, o, cc, t = trial.energy, trial.reward, trial.energy_cost, trial.trial
    fc = transitions_costs[trial.transition][1]
    dv_trials.append(dv[task.s2i[(e, o, cc, fc, t)]])
data['dv'] = dv_trials

# Compute offer specific decision value regressors:
data['dv_23'] = data['dv'].to_numpy() * (data['is_2'].to_numpy() + data['is_3'].to_numpy())
data['dv_14'] = data['dv'].to_numpy() * (data['is_1'].to_numpy() + data['is_4'].to_numpy())

# Compute categorical regressors that should be somewhat similar to our priors:
# Categorical offer regressor for high and low offer
data['is_12'] = data['is_1'].to_numpy() + data['is_2'].to_numpy()
data['is_34'] = data['is_3'].to_numpy() + data['is_4'].to_numpy()
# Continuous regressor for high and low offer:
data['high_vs_low'] = data['is_34'] - data['is_12']

# Categorical costs regressor
data['is_lc'] = (data['energy_cost'] == 1).astype(int).to_numpy()
data['is_hc'] = (data['energy_cost'] == 2).astype(int).to_numpy()
# Continuous costs regressor
data['hc_vs_lc'] = data['is_hc'] - data['is_lc']

# Categorical transition regressor
data['is_trans1'] = (data['transition'] == 0).to_numpy()
data['is_trans2'] = (data['transition'] == 1).to_numpy()
data['is_trans3'] = (data['transition'] == 2).to_numpy()
data['is_trans4'] = (data['transition'] == 3).to_numpy()
# Continuous transition regressor
data['transition_centered'] = data['transition'] - 2.5

# Categorical energy regressor:
data['e_is_0'] = (data['energy'] == 0).to_numpy()
data['e_is_1'] = (data['energy'] == 1).to_numpy()
data['e_is_2'] = (data['energy'] == 2).to_numpy()
data['e_is_3'] = (data['energy'] == 3).to_numpy()
data['e_is_4'] = (data['energy'] == 4).to_numpy()
data['e_is_5'] = (data['energy'] == 5).to_numpy()
data['e_is_6'] = (data['energy'] == 6).to_numpy()

# Contribution of each experimental factor
Our results indicate that participants preference reflect the structure of the task, their bias reflect some form of combination of preferences associated with the levels of each factors. One interesting question is whether participants biases depend more on specific factors than other. Based on Ott's results, we would expect that offer shape participants preferences quite a lot. To explore such effects, we will perform a simple variance partitioning analysis, comparing the amount of variance explained by each experimental factors. For simplicity, we won't consider the interaction between preferences and decision values:

In [ ]:
r2_results = {}

# Fit the full model for reference:
preference_model = bmb.Model(
    "response ~ dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + is_trans1 + is_trans2 + is_trans3 + is_trans4 + " \
    "e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6 + "
    "(dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + is_trans1 + is_trans2 + is_trans3 + is_trans4 +e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6|vpn)", 
    data, 
    family="bernoulli")
idata = preference_model.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)
ref_r2 = preference_model.r2_score(idata)

# Fit the preference model without offer preferences:
preference_model_no_offers = bmb.Model(
    "response ~ dv + is_lc + is_hc + is_trans1 + is_trans2 + is_trans3 + is_trans4 + " \
    "e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6 + "
    "(dv + is_lc + is_hc + is_trans1 + is_trans2 + is_trans3 + is_trans4 +e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6|vpn)", 
    data, 
    family="bernoulli")
idata = preference_model_no_offers.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)
r2_results['preference_model_no_offers'] = preference_model.r2_score(idata)

# Fit the full model without costs preferences:
preference_model_no_costs = bmb.Model(
    "response ~ dv + is_1 + is_2 + is_3 + is_4 + is_trans1 + is_trans2 + is_trans3 + is_trans4 + " \
    "e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6 + "
    "(dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + is_trans1 + is_trans2 + is_trans3 + is_trans4 +e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6|vpn)", 
    data, 
    family="bernoulli")
idata = preference_model_no_costs.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)
r2_results['preference_model_no_costs'] = preference_model.r2_score(idata)

# Fit the full model without transition preferences:
preference_model_no_transitions = bmb.Model(
    "response ~ dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + " \
    "e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6 + "
    "(dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6|vpn)", 
    data, 
    family="bernoulli")
idata = preference_model_no_transitions.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)
r2_results['preference_model_no_transitions'] = preference_model.r2_score(idata)

# Fit the full model without energy preferences:
preference_model_no_energy = bmb.Model(
    "response ~ dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + " \
    "(dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc|vpn)", 
    data, 
    family="bernoulli")
idata = preference_model_no_energy.fit(
    draws=1000, target_accept=0.85, idata_kwargs={"log_likelihood": True},
)
r2_results['preference_model_no_energy'] = preference_model.r2_score(idata)

Modeling the probability that response==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, dv, is_1, is_2, is_3, is_4, is_lc, is_hc, is_trans1, is_trans2, is_trans3, is_trans4, e_is_0, e_is_1, e_is_2, e_is_3, e_is_4, e_is_5, e_is_6, 1|vpn_sigma, 1|vpn_offset, dv|vpn_sigma, dv|vpn_offset, is_1|vpn_sigma, is_1|vpn_offset, is_2|vpn_sigma, is_2|vpn_offset, is_3|vpn_sigma, is_3|vpn_offset, is_4|vpn_sigma, is_4|vpn_offset, is_lc|vpn_sigma, is_lc|vpn_offset, is_hc|vpn_sigma, is_hc|vpn_offset, is_trans1|vpn_sigma, is_trans1|vpn_offset, is_trans2|vpn_sigma, is_trans2|vpn_offset, is_trans3|vpn_sigma, is_trans3|vpn_offset, is_trans4|vpn_sigma, is_trans4|vpn_offset, e_is_0|vpn_sigma, e_is_0|vpn_offset, e_is_1|vpn_sigma, e_is_1|vpn_offset, e_is_2|vpn_sigma, e_is_2|vpn_offset, e_is_3|vpn_sigma, e_is_3|vpn_offset, e_is_4|vpn_sigma, e_is_4|vpn_offset, e_is_5|vpn_sigma, e_is_5|vpn_offset, e_is_6|vpn_sigma, e_is_6|vpn_offset]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 570 seconds.


In [ ]:
# Plot the results:
fig, ax = plt.subplots()
ax.bar(['Offer', 'Costs', 'Transitions', 'Energy'], [r2_results[mdl].r2 - ref_r2 for mdl in r2_results.keys()])
# Add the error bars:
ax.errorbar(['Offer', 'Costs', 'Transitions', 'Energy'], [r2_results[mdl].r2 - ref_r2 for mdl in r2_results.keys()], 
            yerr=[r2_results[mdl].r2_std for mdl in r2_results.keys()], 
            color='k', capsize=5)
ax.set_ylabel("R2 difference with full model")
plt.show(); 

NameError: name 'plt' is not defined